# Data Quality Audit — YouTube Trending Videos Dataset

Every query below is executed live against the raw CSV with DuckDB. The
outputs saved in this notebook are real query results, not typed-in numbers.

Source: `thedevastator/youtube-trending-videos-dataset` on Kaggle, `youtube.csv`
(161,470 rows, 18 columns, four countries US/CANADA/FRANCE/GB, 2017-11-14 to 2018-06-14).


In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
CSV = '../data/youtube.csv'


## Part 1 — Disproving the `time_frame` field description

The Kaggle page describes `time_frame` as "how long the video trended for".
Three pieces of evidence below show it's actually the video's **publish hour
in UTC**, not a duration.

### Evidence 1: value shape — only 24 clock-hour values

In [2]:
con.sql(f"""
    SELECT COUNT(DISTINCT time_frame) AS n_distinct_time_frame
    FROM read_csv_auto('{CSV}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_distinct_time_frame
0,24


In [3]:
con.sql(f"""
    SELECT time_frame, COUNT(*) AS n_rows
    FROM read_csv_auto('{CSV}')
    GROUP BY time_frame
    ORDER BY time_frame
""").df()

,time_frame,n_rows
0,0:00 to 0:59,4379
1,10:00 to 10:59,3670
2,11:00 to 11:59,4331
3,12:00 to 12:59,5659
4,13:00 to 13:59,7523
5,14:00 to 14:59,9667
6,15:00 to 15:59,12495
7,16:00 to 16:59,16477
8,17:00 to 17:59,15135
9,18:00 to 18:59,10406


### Evidence 2: constant across rows for the same video

`NooW_RbfdWI` trends 38 days in GB, 6 in CANADA, 10 in US, 3 in FRANCE.
Within each country, `time_frame` should only take 1 value if it's a
publish moment rather than a duration that accumulates.

In [4]:
con.sql(f"""
    SELECT publish_country,
           COUNT(DISTINCT strptime(trending_date, '%y.%d.%m')::DATE) AS days,
           COUNT(DISTINCT time_frame) AS distinct_time_frame_values
    FROM read_csv_auto('{CSV}')
    WHERE video_id = 'NooW_RbfdWI'
    GROUP BY publish_country
""").df()

,publish_country,days,distinct_time_frame_values
0,FRANCE,3,1
1,US,10,1
2,GB,38,1
3,CANADA,6,1


In [5]:
con.sql(f"""
    SELECT COUNT(*) AS n_videos_with_time_frame_variation
    FROM (
        SELECT video_id
        FROM read_csv_auto('{CSV}')
        GROUP BY video_id
        HAVING COUNT(DISTINCT time_frame) > 1
    )
""").df()

,n_videos_with_time_frame_variation
0,12


### Evidence 3: direct counter-example

Among US-trending videos with `time_frame='0:00 to 0:59'` (which under the
"duration" reading means "trended for under an hour"), the top result
below trended for 22 consecutive days.

In [6]:
con.sql(f"""
    SELECT video_id,
           COUNT(DISTINCT strptime(trending_date, '%y.%d.%m')::DATE) AS days_on_trending
    FROM read_csv_auto('{CSV}')
    WHERE publish_country = 'US' AND time_frame = '0:00 to 0:59'
    GROUP BY video_id
    ORDER BY days_on_trending DESC
    LIMIT 5
""").df()

,video_id,days_on_trending
0,XdNOI-q70q4,22
1,QDk-xa1oBXw,21
2,jxWJLs7_doc,19
3,JyG5hdbQpDM,18
4,L87HuKmGwVQ,18


**Conclusion**: `time_frame` is a publish hour (UTC), not a duration.

## Part 2 — Two data invariants

### Invariant 1: a video can only have one row per country per day

Naively trusting `MAX(COUNT(DISTINCT trending_date))` to find the
longest-lived videos gets polluted by a corrupted ID (`#NAME?`) that fakes
a 191-day "lifespan". Writing this invariant down catches the actual
violations first.

In [7]:
violations = con.sql(f"""
    SELECT video_id, publish_country,
           strptime(trending_date, '%y.%d.%m')::DATE AS trend_dt,
           COUNT(*) AS rows_that_day,
           COUNT(DISTINCT title) AS distinct_titles,
           COUNT(DISTINCT views) AS distinct_views
    FROM read_csv_auto('{CSV}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Distinct violating video_id: {violations['video_id'].nunique()}")
print(f"Total (video_id, country, day) groups: {len(violations)}")
violations.head()

Distinct violating video_id: 180
Total (video_id, country, day) groups: 729


,video_id,publish_country,trend_dt,rows_that_day,distinct_titles,distinct_views
0,#NAME?,GB,2018-05-12,2,2,2
1,RUCXD3_wW2w,GB,2018-05-14,2,1,1
2,p8npDG2ulKQ,GB,2018-05-14,2,1,1
3,aixso4N2vhI,GB,2018-05-14,2,1,1
4,bu0m_UdtoaU,GB,2018-05-14,2,1,1


Split the violations by shape — is it a genuine ID collision or a duplicate same-day scrape?

In [8]:
con.sql(f"""
    SELECT
        CASE WHEN video_id = '#NAME?' THEN 'ID collision (#NAME?)' ELSE 'duplicate same-day scrape' END AS violation_form,
        COUNT(DISTINCT video_id) AS n_ids,
        COUNT(*) AS n_groups
    FROM (
        SELECT video_id, publish_country,
               strptime(trending_date, '%y.%d.%m')::DATE AS trend_dt
        FROM read_csv_auto('{CSV}')
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
    GROUP BY 1
""").df()

,violation_form,n_ids,n_groups
0,duplicate same-day scrape,179,222
1,ID collision (#NAME?),1,507


In [9]:
con.sql(f"""
    SELECT COUNT(*) AS name_error_rows
    FROM read_csv_auto('{CSV}')
    WHERE video_id = '#NAME?'
""").df()

,name_error_rows
0,1799


### Invariant 2: a video_id's publish attributes must be constant

Invariant 1 only catches same-day collisions. This second invariant catches
"different videos sharing one ID that never collide on the same day" —
`publish_date` and `time_frame` should never vary for a single, correctly
identified video.

In [10]:
con.sql(f"""
    CREATE OR REPLACE TABLE bad_pub AS
    SELECT video_id
    FROM read_csv_auto('{CSV}')
    WHERE video_id <> '#NAME?'
    GROUP BY 1
    HAVING COUNT(DISTINCT publish_date) > 1
        OR COUNT(DISTINCT time_frame) > 1
""")
con.sql("SELECT COUNT(*) AS n_bad_pub_ids FROM bad_pub").df()

,n_bad_pub_ids
0,11


In [11]:
con.sql(f"""
    SELECT COUNT(*) AS n_bad_pub_rows
    FROM read_csv_auto('{CSV}')
    WHERE video_id IN (SELECT video_id FROM bad_pub)
""").df()

,n_bad_pub_rows
0,72


## Three-step cleaning

1. Isolate `#NAME?` (unrecoverable ID collision)
2. Isolate the 11 IDs with inconsistent publish attributes
3. Dedup same-day duplicate scrapes, keeping the row with the highest views/likes

Expected result: 161,470 - 1,799 - 72 - 222 = **159,377** rows.

In [12]:
con.sql(f"""
    CREATE OR REPLACE TABLE clean AS
    SELECT * EXCLUDE (rn, trend_dt), trend_dt FROM (
        SELECT *,
               strptime(trending_date, '%y.%d.%m')::DATE AS trend_dt,
               ROW_NUMBER() OVER (
                   PARTITION BY video_id, publish_country, strptime(trending_date, '%y.%d.%m')::DATE
                   ORDER BY views DESC, likes DESC
               ) AS rn
        FROM read_csv_auto('{CSV}')
        WHERE video_id <> '#NAME?'
          AND video_id NOT IN (SELECT video_id FROM bad_pub)
    ) WHERE rn = 1
""")
con.sql("SELECT COUNT(*) AS n_clean_rows FROM clean").df()

,n_clean_rows
0,159377


## Scrape completeness

Must run **after** cleaning/deduping — duplicate rows would mask a
partial-capture day.

In [13]:
con.sql("""
    SELECT publish_country,
           COUNT(*) AS n_days,
           MAX(trend_dt) - MIN(trend_dt) + 1 AS trending_span_days,
           MEDIAN(n_rows) AS median_rows_per_day,
           MIN(n_rows) AS min_rows_per_day
    FROM (
        SELECT publish_country, trend_dt, COUNT(*) AS n_rows
        FROM clean
        GROUP BY 1, 2
    )
    GROUP BY 1
""").df()

,publish_country,n_days,trending_span_days,median_rows_per_day,min_rows_per_day
0,US,205,213,198.0,147
1,CANADA,205,213,197.0,169
2,FRANCE,205,213,197.0,164
3,GB,205,213,197.0,73


In [14]:
missing_days = con.sql("""
    SELECT c.publish_country, cal.d AS missing_day
    FROM (SELECT DISTINCT publish_country FROM clean) c
    CROSS JOIN (
        SELECT UNNEST(generate_series(DATE '2017-11-14', DATE '2018-06-14', INTERVAL 1 DAY))::DATE AS d
    ) cal
    LEFT JOIN (SELECT DISTINCT publish_country, trend_dt FROM clean) have
        ON have.publish_country = c.publish_country AND have.trend_dt = cal.d
    WHERE have.trend_dt IS NULL
    ORDER BY 1, 2
""").df()
print(f"Total missing (country, day) rows: {len(missing_days)}")
missing_days

Total missing (country, day) rows: 32


,publish_country,missing_day
0,CANADA,2018-01-10
1,CANADA,2018-01-11
2,CANADA,2018-04-08
3,CANADA,2018-04-09
4,CANADA,2018-04-10
5,CANADA,2018-04-11
6,CANADA,2018-04-12
7,CANADA,2018-04-13
8,FRANCE,2018-01-10
9,FRANCE,2018-01-11


In [15]:
con.sql("""
    SELECT publish_country, trend_dt, COUNT(*) AS n_rows
    FROM clean
    GROUP BY 1, 2
    ORDER BY n_rows ASC
    LIMIT 5
""").df()

,publish_country,trend_dt,n_rows
0,GB,2018-05-15,73
1,GB,2018-05-14,114
2,GB,2018-05-16,147
3,US,2018-05-15,147
4,GB,2018-05-22,151


**Conclusion**: all four countries are missing the exact same 8 days
(2018-01-10, 01-11, 04-08 through 04-13) — a synchronized gap across four
independent markets means the upstream scraper went down, not that four
trending lists happened to be empty at once. GB's 2018-05-15 had only 73
rows — a partial capture, confirmed above.

## Data quality checklist — attribute drift and other issues

Computed on `clean`, not raw — otherwise `#NAME?` and the bad-publish IDs
would pollute these counts.

In [16]:
con.sql("""
    SELECT
        (SELECT COUNT(*) FROM (SELECT video_id FROM clean GROUP BY 1 HAVING COUNT(DISTINCT channel_title) > 1)) AS channel_title_changed,
        (SELECT COUNT(*) FROM (SELECT video_id FROM clean GROUP BY 1 HAVING COUNT(DISTINCT title) > 1)) AS title_changed,
        (SELECT COUNT(*) FROM (SELECT video_id FROM clean GROUP BY 1 HAVING COUNT(DISTINCT category_id) > 1)) AS category_id_changed
""").df()

,channel_title_changed,title_changed,category_id_changed
0,55,538,49


In [17]:
con.sql("""
    SELECT COUNT(*) AS views_regressed_rows
    FROM (
        SELECT views,
               LAG(views) OVER (PARTITION BY video_id, publish_country ORDER BY trend_dt) AS prev_views
        FROM clean
    )
    WHERE views < prev_views
""").df()

,views_regressed_rows
0,100


In [18]:
con.sql("""
    SELECT COUNT(*) AS video_error_rows
    FROM clean
    WHERE video_error_or_removed = True
""").df()

,video_error_rows
0,126


In [19]:
con.sql("""
    SELECT COUNT(*) AS channel_title_case_variant_groups
    FROM (
        SELECT LOWER(TRIM(channel_title)) AS norm_title, COUNT(DISTINCT channel_title) AS variants
        FROM clean
        GROUP BY 1
        HAVING COUNT(DISTINCT channel_title) > 1
    )
""").df()

,channel_title_case_variant_groups
0,53


In [20]:
con.sql("SELECT COUNT(DISTINCT category_id) AS n_distinct_categories FROM clean").df()

,n_distinct_categories
0,18


## Summary

| Check | Result |
|---|---|
| `time_frame` distinct values | 24 (publish hour, not duration) |
| Invariant 1 violations | 180 distinct video_id, 729 groups |
| `#NAME?` rows (isolated) | 1,799 |
| Invariant 2 violations (isolated) | 11 IDs, 72 rows |
| Rows after three-step cleaning | 159,377 |
| Days missing across all 4 countries | 8 |
| Partial-capture day | GB, 2018-05-15, 73 rows |
| Videos with channel name drift | 55 |
| Videos with title drift | 538 |
| Videos with category drift | 49 |
| Rows with regressed views | 100 |
| Rows with video_error_or_removed | 126 |
| channel_title case/whitespace variant groups | see output above |
| Distinct category_id values | 18 (no name mapping) |
